# DRC benchmark — full local-model runs on Kaggle

Run **one model at a time** in this order: NLLB, BLOOMZ, MADLAD, Apertus. Each session writes a resumable private prediction file and stops cleanly after ten inference hours so its output can be saved. Raw benchmark text and predictions must remain private.

## Kaggle setup

1. Enable a GPU and Internet in Notebook settings.
2. Add the private benchmark ZIP as a **private** notebook input.
3. For a resumed session, also attach the preceding notebook output containing `predictions.jsonl`.
4. Change only `MODEL_KEY` below. Use `nllb`, then `bloomz`, `madlad`, and `apertus`.
5. After the run stops, use **Save Version** so `/kaggle/working` becomes an attachable private output for the next session.

In [ ]:
MODEL_KEY = "nllb"  # nllb | bloomz | madlad | apertus
BATCH_SIZE = 16         # OOM batches are automatically divided
SESSION_MINUTES = 600   # leaves time to archive and save before Kaggle ends
REPO_URL = "https://github.com/Ashuza11/CongoLangBench.git"
REPO_BRANCH = "main"
assert MODEL_KEY in {"nllb", "bloomz", "madlad", "apertus"}

In [ ]:
%pip install -q -U "transformers>=4.56,<5" accelerate bitsandbytes sentencepiece protobuf sacrebleu "pandas==2.2.3"

In [ ]:
import platform, subprocess, torch
from pathlib import Path
if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU before continuing.")
gpu = torch.cuda.get_device_properties(0)
print(f"Python {platform.python_version()} | PyTorch {torch.__version__}")
print(f"GPU: {gpu.name} ({gpu.total_memory / 2**30:.1f} GiB)")

In [ ]:
REPO_ROOT = Path("/kaggle/working/CongoLangBench")
if REPO_ROOT.exists():
    subprocess.run(["git", "-C", str(REPO_ROOT), "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, "--depth", "1", REPO_URL, str(REPO_ROOT)], check=True)
print(subprocess.check_output(["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"], text=True).strip())

In [ ]:
import shutil, zipfile
input_root = Path("/kaggle/input")
benchmark_zips = list(input_root.rglob("congolang-benchmark-v1.zip"))
if len(benchmark_zips) != 1:
    raise FileNotFoundError(f"Attach exactly one private benchmark ZIP; found {benchmark_zips}")
DATA_ROOT = Path("/kaggle/working/congolang-benchmark-private")
if DATA_ROOT.exists():
    shutil.rmtree(DATA_ROOT)
DATA_ROOT.mkdir(parents=True)
with zipfile.ZipFile(benchmark_zips[0]) as archive:
    for member in archive.infolist():
        destination = (DATA_ROOT / member.filename).resolve()
        if not destination.is_relative_to(DATA_ROOT.resolve()):
            raise ValueError(f"Unsafe ZIP member: {member.filename}")
    archive.extractall(DATA_ROOT)
print(f"Extracted private benchmark from {benchmark_zips[0]}")

In [ ]:
# Find a preceding private checkpoint only for the selected model.
resume_candidates = [
    path for path in input_root.rglob("predictions.jsonl")
    if MODEL_KEY in str(path).lower()
]
if len(resume_candidates) > 1:
    raise ValueError(f"Attach at most one {MODEL_KEY} checkpoint: {resume_candidates}")
RESUME_PATH = resume_candidates[0] if resume_candidates else None
print(f"Resume checkpoint: {RESUME_PATH or 'none — starting this model'}")

## Run or resume production inference

This is the full evaluation, not a pilot. NLLB and MADLAD automatically write `coverage.json` and skip tracks without exact model language tags.

In [ ]:
OUTPUT_ROOT = Path(f"/kaggle/working/{MODEL_KEY}-full-v1")
command = [
    "python", "-u", str(REPO_ROOT / "scripts/run_kaggle_model.py"),
    "--model-key", MODEL_KEY,
    "--repo-root", str(REPO_ROOT),
    "--data-root", str(DATA_ROOT),
    "--output-root", str(OUTPUT_ROOT),
    "--batch-size", str(BATCH_SIZE),
    "--max-runtime-minutes", str(SESSION_MINUTES),
]
if RESUME_PATH:
    command.extend(["--resume-predictions", str(RESUME_PATH)])
subprocess.run(command, check=True)

In [ ]:
metadata_path = OUTPUT_ROOT / "run_metadata.json"
if metadata_path.exists():
    score_command = [
        "python", str(REPO_ROOT / "scripts/score_full_run.py"),
        "--repo-root", str(REPO_ROOT),
        "--data-root", str(DATA_ROOT),
        "--run-root", str(OUTPUT_ROOT),
        "--output-root", str(OUTPUT_ROOT / "scored"),
    ]
    subprocess.run(score_command, check=True)
archive = shutil.make_archive(
    f"/kaggle/working/{MODEL_KEY}-full-v1-private-checkpoint", "zip", OUTPUT_ROOT
)
if metadata_path.exists():
    print(f"{MODEL_KEY} and aggregate scoring are COMPLETE. Private archive: {archive}")
else:
    print(f"{MODEL_KEY} is INCOMPLETE but safely checkpointed: {archive}")
    print("Save this notebook version, attach its output privately next time, and rerun with the same MODEL_KEY.")